# Edge-Cloud Offloading: Evaluation & Analysis

In this notebook, you will analyze the offloading results from your gesture recognition system. You will sweep confidence thresholds, compare local vs. cloud accuracy, visualize latency distributions, and estimate bandwidth costs.

**Your tasks:**
- Run each cell in order
- Cells marked `# YOUR CODE HERE` require you to write code
- Answer each **Checkpoint** question in the markdown cell provided
- Use **Kernel > Restart & Run All** if you update your results CSV

**Prerequisites:** Run `test_offloading.py` first to generate `offloading_results.csv`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
    accuracy_score,
)

sns.set_theme(style="whitegrid")
%matplotlib inline

---
## Section 1: Load & Explore Results

First, let's load the results CSV generated by `test_offloading.py` and understand the dataset.

In [ ]:
# Load results
df = pd.read_csv("offloading_results.csv")

print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
df.head(10)

In [ ]:
# Label distribution
print("True label distribution:")
print(df["true_label"].value_counts())

print(f"\nBasic latency statistics:")
print(f"  Local  - mean: {df['local_latency_ms'].mean():.1f} ms, "
      f"std: {df['local_latency_ms'].std():.1f} ms")
print(f"  Cloud  - mean: {df['cloud_latency_ms'].mean():.1f} ms, "
      f"std: {df['cloud_latency_ms'].std():.1f} ms")

print(f"\nConfidence statistics:")
print(f"  Local  - mean: {df['local_confidence'].mean():.1f}%, "
      f"min: {df['local_confidence'].min():.1f}%, "
      f"max: {df['local_confidence'].max():.1f}%")
print(f"  Cloud  - mean: {df['cloud_confidence'].mean():.1f}%, "
      f"min: {df['cloud_confidence'].min():.1f}%, "
      f"max: {df['cloud_confidence'].max():.1f}%")

### Checkpoint 1

**What is the ratio of cloud latency to local latency? Is it consistent across samples?**

*Your answer here:*

---
## Section 2: Threshold Sweep & Offload Rate

The confidence threshold determines when the ESP32 trusts its own prediction vs. offloading to the cloud. Let's sweep across thresholds to understand the tradeoffs.

**Note on hybrid latency:** When the ESP32 offloads, it first runs local inference to check confidence, then sends data to the cloud. So hybrid latency for offloaded samples is `local_latency + cloud_latency` (both steps happen sequentially).

In [ ]:
# Sweep thresholds from 50% to 95%
thresholds = np.arange(50, 96, 5)
sweep_results = []

for threshold in thresholds:
    # For each sample, decide: use local or cloud prediction?
    hybrid_predictions = []
    hybrid_latencies = []
    offloaded_count = 0

    for _, row in df.iterrows():
        if row["local_confidence"] >= threshold:
            # Trust local prediction
            hybrid_predictions.append(row["local_prediction"])
            hybrid_latencies.append(row["local_latency_ms"])
        else:
            # Offload to cloud
            hybrid_predictions.append(row["cloud_prediction"])
            hybrid_latencies.append(
                row["local_latency_ms"] + row["cloud_latency_ms"]
            )
            offloaded_count += 1

    offload_rate = offloaded_count / len(df) * 100
    hybrid_accuracy = accuracy_score(df["true_label"], hybrid_predictions) * 100
    local_accuracy = accuracy_score(df["true_label"], df["local_prediction"]) * 100
    cloud_accuracy = accuracy_score(df["true_label"], df["cloud_prediction"]) * 100
    avg_latency = np.mean(hybrid_latencies)

    sweep_results.append({
        "threshold": threshold,
        "offload_rate": offload_rate,
        "hybrid_accuracy": hybrid_accuracy,
        "local_accuracy": local_accuracy,
        "cloud_accuracy": cloud_accuracy,
        "avg_latency_ms": avg_latency,
    })

sweep_df = pd.DataFrame(sweep_results)
sweep_df

In [ ]:
# Three-panel plot
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1: Offload rate vs. threshold
axes[0].plot(sweep_df["threshold"], sweep_df["offload_rate"], "o-", color="#2196F3")
axes[0].set_xlabel("Confidence Threshold (%)")
axes[0].set_ylabel("Offload Rate (%)")
axes[0].set_title("Offload Rate vs. Threshold")
axes[0].grid(True, alpha=0.3)

# Panel 2: Accuracy vs. threshold
axes[1].plot(sweep_df["threshold"], sweep_df["hybrid_accuracy"], "s-",
             color="#4CAF50", label="Hybrid")
axes[1].axhline(y=sweep_df["local_accuracy"].iloc[0], color="#FF9800",
                linestyle="--", label="Local-only")
axes[1].axhline(y=sweep_df["cloud_accuracy"].iloc[0], color="#9C27B0",
                linestyle="--", label="Cloud-only")
axes[1].set_xlabel("Confidence Threshold (%)")
axes[1].set_ylabel("Accuracy (%)")
axes[1].set_title("Accuracy vs. Threshold")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Panel 3: Latency vs. threshold
axes[2].plot(sweep_df["threshold"], sweep_df["avg_latency_ms"], "^-", color="#F44336")
axes[2].set_xlabel("Confidence Threshold (%)")
axes[2].set_ylabel("Avg Latency (ms)")
axes[2].set_title("Average Latency vs. Threshold")
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# YOUR CODE HERE: Choose your optimal confidence threshold based on the plots above.
# Consider: at what point does hybrid accuracy exceed local-only accuracy,
# and what is the latency cost of that improvement?

chosen_threshold = None  # Replace None with your chosen threshold (e.g., 75)

assert chosen_threshold is not None, "Set chosen_threshold to a number!"
print(f"Chosen threshold: {chosen_threshold}%")

# Show metrics at chosen threshold
chosen_row = sweep_df[sweep_df["threshold"] == chosen_threshold]
if len(chosen_row) == 0:
    # Interpolate if exact threshold not in sweep
    print("Threshold not in sweep range. Pick from:", list(sweep_df['threshold']))
else:
    row = chosen_row.iloc[0]
    print(f"  Offload rate:     {row['offload_rate']:.1f}%")
    print(f"  Hybrid accuracy:  {row['hybrid_accuracy']:.1f}%")
    print(f"  Local-only acc:   {row['local_accuracy']:.1f}%")
    print(f"  Cloud-only acc:   {row['cloud_accuracy']:.1f}%")
    print(f"  Avg latency:      {row['avg_latency_ms']:.1f} ms")

### Checkpoint 2

**At what threshold does the hybrid approach first outperform pure-local inference? What is the latency tradeoff at that point?**

*Your answer here:*

---
## Section 3: Confusion Matrices

Let's compare per-class performance across three strategies: local-only, cloud-only, and hybrid at your chosen threshold.

In [ ]:
# Build hybrid predictions at chosen threshold
hybrid_preds = []
for _, row in df.iterrows():
    if row["local_confidence"] >= chosen_threshold:
        hybrid_preds.append(row["local_prediction"])
    else:
        hybrid_preds.append(row["cloud_prediction"])

labels = sorted(df["true_label"].unique())

# Side-by-side confusion matrices
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

strategies = {
    "Local-Only": df["local_prediction"],
    "Cloud-Only": df["cloud_prediction"],
    f"Hybrid (threshold={chosen_threshold}%)": hybrid_preds,
}

for idx, (name, preds) in enumerate(strategies.items()):
    cm = confusion_matrix(df["true_label"], preds, labels=labels)
    ConfusionMatrixDisplay(cm, display_labels=labels).plot(ax=axes[idx], cmap="Blues")
    axes[idx].set_title(name)

plt.suptitle("Confusion Matrices by Strategy", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Classification reports
for name, preds in strategies.items():
    print(f"\n{'='*55}")
    print(f"{name}")
    print(f"{'='*55}")
    print(classification_report(df["true_label"], preds, labels=labels))

### Checkpoint 3

**Which gesture class benefits most from cloud offloading? Report the F1 score difference between local-only and hybrid for each class.**

*Your answer here:*

---
## Section 4: Latency Distribution Analysis

Understanding latency distributions is critical for real-time applications. Let's visualize how local and cloud inference latencies compare.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram overlay
axes[0].hist(df["local_latency_ms"], bins=20, alpha=0.6, label="Local", color="#4CAF50")
axes[0].hist(df["cloud_latency_ms"], bins=20, alpha=0.6, label="Cloud", color="#2196F3")
axes[0].set_xlabel("Latency (ms)")
axes[0].set_ylabel("Count")
axes[0].set_title("Latency Distribution: Local vs. Cloud")
axes[0].legend()

# Box plot comparison
latency_data = pd.DataFrame({
    "Local": df["local_latency_ms"],
    "Cloud": df["cloud_latency_ms"],
})
latency_data.boxplot(ax=axes[1])
axes[1].set_ylabel("Latency (ms)")
axes[1].set_title("Latency Box Plot: Local vs. Cloud")

plt.tight_layout()
plt.show()

In [ ]:
# Summary statistics table
latency_stats = pd.DataFrame({
    "Metric": ["Mean", "Median", "Std Dev", "P95", "P99"],
    "Local (ms)": [
        f"{df['local_latency_ms'].mean():.1f}",
        f"{df['local_latency_ms'].median():.1f}",
        f"{df['local_latency_ms'].std():.1f}",
        f"{df['local_latency_ms'].quantile(0.95):.1f}",
        f"{df['local_latency_ms'].quantile(0.99):.1f}",
    ],
    "Cloud (ms)": [
        f"{df['cloud_latency_ms'].mean():.1f}",
        f"{df['cloud_latency_ms'].median():.1f}",
        f"{df['cloud_latency_ms'].std():.1f}",
        f"{df['cloud_latency_ms'].quantile(0.95):.1f}",
        f"{df['cloud_latency_ms'].quantile(0.99):.1f}",
    ],
})
print("Latency Summary")
print("=" * 40)
print(latency_stats.to_string(index=False))

### Checkpoint 4

**What is the P95 cloud latency? If your application requires responses in under 500ms, what is the maximum offload rate you can sustain?**

*Your answer here:*

---
## Section 5: Bandwidth & Cost Estimation

Offloading to the cloud sends raw sensor data over the network. Let's estimate the bandwidth impact.

In [ ]:
# Given parameters
FEATURE_SIZE = 300               # Number of floats per gesture
BYTES_PER_FLOAT = 4              # 32-bit float
JSON_OVERHEAD_FACTOR = 2.5       # JSON text is ~2.5x larger than raw binary
GESTURES_PER_MINUTE = 10         # Assumed usage rate
ACTIVE_HOURS_PER_DAY = 8         # Hours of active use per day
DAYS_PER_MONTH = 30

# Size of one feature vector as JSON payload (bytes)
payload_size_bytes = FEATURE_SIZE * BYTES_PER_FLOAT * JSON_OVERHEAD_FACTOR
print(f"Payload size per gesture: {payload_size_bytes:.0f} bytes ({payload_size_bytes/1024:.2f} KB)")

In [ ]:
# YOUR CODE HERE: Calculate monthly bandwidth at your chosen threshold.
#
# Use these variables:
#   - chosen_row["offload_rate"].values[0] gives the offload rate (%) at your threshold
#   - GESTURES_PER_MINUTE, ACTIVE_HOURS_PER_DAY, DAYS_PER_MONTH
#   - payload_size_bytes
#
# Steps:
# 1. Calculate total gestures per month
# 2. Calculate offloaded gestures per month (using offload rate)
# 3. Calculate total bandwidth in bytes, then convert to MB and GB

offload_rate_fraction = None  # e.g., 0.45 for 45% offload rate
total_gestures_per_month = None
offloaded_gestures_per_month = None
monthly_bandwidth_bytes = None
monthly_bandwidth_mb = None
monthly_bandwidth_gb = None

# Uncomment and fill in:
# offload_rate_fraction = ...
# total_gestures_per_month = ...
# offloaded_gestures_per_month = ...
# monthly_bandwidth_bytes = ...
# monthly_bandwidth_mb = ...
# monthly_bandwidth_gb = ...

assert monthly_bandwidth_mb is not None, "Complete the bandwidth calculation!"

print(f"Offload rate: {offload_rate_fraction * 100:.1f}%")
print(f"Total gestures/month: {total_gestures_per_month:,.0f}")
print(f"Offloaded gestures/month: {offloaded_gestures_per_month:,.0f}")
print(f"Monthly bandwidth: {monthly_bandwidth_mb:.1f} MB ({monthly_bandwidth_gb:.3f} GB)")

### Checkpoint 5

**Is this bandwidth significant for a cellular IoT deployment? For a WiFi deployment? Justify your answer with numbers.**

*Your answer here:*

---
## Section 6: Summary

Let's compile a final summary of your offloading system analysis.

In [ ]:
# Final summary table
chosen_row_data = sweep_df[sweep_df["threshold"] == chosen_threshold].iloc[0]

summary = pd.DataFrame({
    "Metric": [
        "Confidence Threshold",
        "Offload Rate",
        "Local-Only Accuracy",
        "Cloud-Only Accuracy",
        "Hybrid Accuracy",
        "Avg Local Latency",
        "Avg Cloud Latency",
        "Avg Hybrid Latency",
        "Monthly Bandwidth",
    ],
    "Value": [
        f"{chosen_threshold}%",
        f"{chosen_row_data['offload_rate']:.1f}%",
        f"{chosen_row_data['local_accuracy']:.1f}%",
        f"{chosen_row_data['cloud_accuracy']:.1f}%",
        f"{chosen_row_data['hybrid_accuracy']:.1f}%",
        f"{df['local_latency_ms'].mean():.1f} ms",
        f"{df['cloud_latency_ms'].mean():.1f} ms",
        f"{chosen_row_data['avg_latency_ms']:.1f} ms",
        f"{monthly_bandwidth_mb:.1f} MB",
    ],
})

print("Final Offloading System Summary")
print("=" * 45)
print(summary.to_string(index=False))

### Checkpoint 6

**In one paragraph, recommend whether to deploy this offloading system in production. Cite at least 3 quantitative findings from your analysis (e.g., accuracy improvement, latency cost, bandwidth).**

*Your answer here:*